## Altitude adjusted data
The idea here is to adjust all temperatures based on the scaling their elevations to the mean elevation of the smallest station category $z_b$. Functionally this means finding the difference between each station and the average urban station elevation. For simulated observations at station points we therefore want to use the blurred orography +2m to represent $z_b$. After this we can take a constant lapse rate to adjust each temperature to the mean elevation.

$$\frac{\partial T}{\partial z} = \frac{-g}{c_p}$$

So the adjusted temperature $T_a$ for each station given some unadjusted temperature $T_b$ at elevation $z_b$ follows

$$T_a = T_b + \frac{g}{c_p} (z_b - z_a) $$

By using the mean elevation we minimise the reliance on a constant lapse rate assumptions without entirely ignoring differences that small changes in elevation can have on temperature. Since elevation outlisers have already been filtered from the obs dataset, this new set should help minimise elevation even further as a confounding factor to disentangle from the UHI and all accuracy/plots should be regenerated with this in mind.  



In [1]:
from Montreal_UHI_toolbox import obs, obs_rural
import numpy as np
from collections.abc import Iterable

/runoff/gulley/.miniconda3/lib/python3.12/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.39.0 or higher is recommended. You are running version 2.14.1
  warnings.warn(


In [ ]:
Z_a = np.mean(obs_rural.elev.values) # mean elevation of the smallest dataset to which all temperatures should be adjusted
def adjust_temp(T_b, z_b, z_a=Z_a):
    """
    Adjust temperature series to a target elevation using a constant lapse rate.

    Adjusts an input temperature series to a specified elevation using the dry adiabatic lapse rate
    (g/c_p), where g is the gravitational acceleration and c_p is the specific heat capacity of air
    at constant pressure.
    Parameters
    ----------
    T_b : array-like of float
        Recorded temperature before adjustment in Kelvin.
    z_b : array-like of float
        Recorded elevation before adjustment in meters.
    z_a : float, optional
        Elevation to which the temperature is adjusted in meters. Default is the mean elevation
        of the smallest dataset (Z_a).
    Returns
    -------
    T_a : array-like of float
        Adjusted temperature in Kelvin, accounting for elevation differences using the constant
        lapse rate (g/c_p).

    Notes
    -----
    If adjusting based on orography, manually add 2 meters to the elevation values.

    Examples
    --------
    >>> T_b = [[280.0, 285.0, 282.5], # Temperature series at 1st station
    >>>        [279.0, 281.0, 286.0]] # Temperature series at 2nd station
    >>> z_b = [0, 100]                # Elevations at 1st and 2nd station respectively
    >>> z_a = 50.0                    # Mean elevation to adjust temperatures to
    >>> adjust_temp(T_b, z_b, z_a)    # Follows format of T_b
    array([[279.51262425, 284.51262425, 282.01262425],
           [279.48737575, 281.48737575, 286.48737575]])
    """
    g = 9.806
    c_p = 1006

    if isinstance(T_b, Iterable):
        elev_diff = z_b - z_a * np.ones(np.shape(z_b))
        T_a = (g/c_p) * elev_diff.reshape(-1, 1) + T_b
    else:
        T_a = (g/c_p) * (z_b - z_a)
    return T_a


In [43]:
z_b = obs.elev.values
T_b = obs.tasmin.values + 273.15
T_a = adjust_temp(T_b,z_b)

In [44]:
T_a

array([[258.12693951, 265.22693951, 266.82693951, ..., 269.52693951,
        274.42693951, 274.92693951],
       [261.50926555, 269.50926555, 271.50926555, ..., 269.10926555,
        274.30926555, 277.50926555],
       [257.28507271, 269.28507271, 270.28507271, ..., 270.98507271,
        276.78507271, 277.58507271],
       ...,
       [257.94390968, 269.94390968, 268.94390968, ..., 269.74390968,
        276.64390968, 276.94390968],
       [264.39986992, 269.89986992, 267.99986992, ..., 266.39986992,
        280.59986992, 277.09986992],
       [260.50926555, 271.00926555, 270.50926555, ..., 270.00926555,
        274.20926555, 275.60926555]])

In [45]:
T_b

array([[258.45, 265.55, 267.15, ..., 269.85, 274.75, 275.25],
       [261.15, 269.15, 271.15, ..., 268.75, 273.95, 277.15],
       [257.15, 269.15, 270.15, ..., 270.85, 276.65, 277.45],
       ...,
       [258.15, 270.15, 269.15, ..., 269.95, 276.85, 277.15],
       [264.45, 269.95, 268.05, ..., 266.45, 280.65, 277.15],
       [260.15, 270.65, 270.15, ..., 269.65, 273.85, 275.25]])

In [47]:
T_a

array([[258.12693951, 265.22693951, 266.82693951, ..., 269.52693951,
        274.42693951, 274.92693951],
       [261.50926555, 269.50926555, 271.50926555, ..., 269.10926555,
        274.30926555, 277.50926555],
       [257.28507271, 269.28507271, 270.28507271, ..., 270.98507271,
        276.78507271, 277.58507271],
       ...,
       [257.94390968, 269.94390968, 268.94390968, ..., 269.74390968,
        276.64390968, 276.94390968],
       [264.39986992, 269.89986992, 267.99986992, ..., 266.39986992,
        280.59986992, 277.09986992],
       [260.50926555, 271.00926555, 270.50926555, ..., 270.00926555,
        274.20926555, 275.60926555]])

In [49]:
T_b = [ [280.0, 285.0,282.5] , [279.0,281.0,286.0] ]
z_b = [0, 100]
z_a = 50.0
T_a = adjust_temp(T_b, z_b, z_a)
T_a

array([[279.51262425, 284.51262425, 282.01262425],
       [279.48737575, 281.48737575, 286.48737575]])

In [31]:
T_a - T_b

array([[-0.48737575, -0.48737575],
       [ 0.48737575,  0.48737575]])

In [39]:
elev_diff = z_b - z_a * np.ones(np.shape(z_b))
elev_diff - T_b

array([-330., -235.])

In [37]:
T_b

[280.0, 285.0]